# Single vs. Mutli-Agent System for Utilization Review
Retrieve Patient Record -> Fetch Guidelines -> Perform Eligibility Check -> Make Decision
### What is utilization review?
A utilization review is the evaluation of a patient's care plan to ensure treatments are medically necessary. It often involves heathcare staff cooridinating with insurance providers to approve or suggest appropriate care options

In [ ]:
!pip install langchain==1.2.15 langchain-community==0.4.1 langchain-groq==1.2.2 langgraph==1.1.8 --quiet

# Configure API key

In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

## Imports


In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage ,AIMessage
from langgraph.prebuilt import create_react_agent
from IPython.display import display, Markdown

## Sample Data Overview

This notebook uses simple Python lists (in-memory data) so everything runs easily without needing a database.

### Data Included
1. Medical Guidelines

Rules that help decide if a medical procedure is appropriate.

Fields:
procedure - the test or scan (e.g., CT Abdomen)
diagnosis - the suspected condition
required_symptoms - symptoms that should be present
notes -  extra explanation

```
{
  "procedure": "CT Abdomen",
  "diagnosis": "Suspected Appendicitis",
  "required_symptoms": ["abdominal pain", "nausea", "RLQ tenderness"],
  "notes": "CT imaging justified if appendicitis is unclear."
}```

2. Care Recommendations

Suggested next steps based on diagnosis.

Fields:
diagnosis - condition name
next_step - what to do next



```
{
  "diagnosis": "Community-Acquired Pneumonia",
  "next_step": "Start antibiotics; use CT only if patient is not improving."
}
```

3. Patient Records

Information about individual patients.

Fields:
patient_id
age, sex
symptoms - list of symptoms
diagnosis
procedure - recommended test
notes

```
{
  "patient_id": "P101",
  "age": 38,
  "sex": "Male",
  "symptoms": ["abdominal pain", "nausea"],
  "diagnosis": "Possible early appendicitis",
  "procedure": "CT Abdomen",
  "notes": "Pain present but no clear localized tenderness."
}
```
How the Agent Works
1. Match Guidelines

The agent finds the closest guideline using:
procedure + diagnosis

2. Check Validity

It compares:patient symptoms vs required symptoms in guidelines
Then uses notes for context.

3. Suggest Care Plan

The agent: finds the diagnosis in care_recommendations rewrites the next step in simple language

Example:

“Start antibiotics. Only consider CT scan if the patient does not improve.”


In [ ]:
medical_guidelines = [
    {"procedure": "MRI Brain", "diagnosis": "Migraine", "required_symptoms": ["headache", "nausea"],
     "notes": "MRI not recommended unless neurological deficits or red flags present."},
    {"procedure": "CT Chest", "diagnosis": "Suspected Pulmonary Embolism", "required_symptoms": ["chest pain", "shortness of breath", "tachycardia"],
     "notes": "CTPA appropriate for high probability PE cases with positive D-dimer."},
    {"procedure": "MRI Lumbar Spine", "diagnosis": "Chronic Low Back Pain", "required_symptoms": ["back pain > 6 weeks", "neurological deficit"],
     "notes": "MRI only if pain persists despite conservative therapy and neuro signs are present."},
    {"procedure": "CT Chest", "diagnosis": "Community-Acquired Pneumonia", "required_symptoms": ["fever", "cough"],
     "notes": "CT Chest reserved for inconclusive X-rays or immunocompromised patients."},
    {"procedure": "CT Abdomen", "diagnosis": "Suspected Appendicitis", "required_symptoms": ["abdominal pain", "nausea", "RLQ tenderness"],
     "notes": "CT imaging justified if appendicitis is unclear."}
]

care_recommendations = [
    {"diagnosis": "Migraine", "next_step": "Start migraine treatment; imaging not necessary unless red flags appear."},
    {"diagnosis": "Suspected Pulmonary Embolism", "next_step": "Begin anticoagulation and confirm with CTPA."},
    {"diagnosis": "Chronic Low Back Pain", "next_step": "Refer to physiotherapy; MRI only if neuro symptoms persist."},
    {"diagnosis": "Community-Acquired Pneumonia", "next_step": "Start empirical antibiotics; reserve CT for poor responders."},
    {"diagnosis": "Suspected Appendicitis", "next_step": "Do CT to confirm and refer for surgery if positive."}
]

patient_records = [
    {"patient_id": "P101", "age": 38, "sex": "Male", "symptoms": ["abdominal pain", "nausea"],
     "diagnosis": "Possible early appendicitis", "procedure": "CT Abdomen",
     "notes": "Mild abdominal pain and nausea but no localized tenderness or rebound noted."},
    {"patient_id": "P102", "age": 65, "sex": "Female", "symptoms": ["chest pain", "shortness of breath", "tachycardia"],
     "diagnosis": "Clinical suspicion of PE", "procedure": "CT Chest",
     "notes": "Wells score high probability; D-dimer positive."},
    {"patient_id": "P103", "age": 30, "sex": "Female", "symptoms": ["recurrent headache"],
     "diagnosis": "Classic migraine presentation", "procedure": "MRI Brain",
     "notes": "No neuro signs or red flags. Typical migraine pattern."}
]

## Tools for the Utilization Review Agent

These tools (defined with @tool) are used by the agent during a review.
Each tool handles a specific piece of logic and returns a small, structured dictionary that the agent can interpret.

### Tool Summary

**fetch_patient_record**

Purpose: Retrieve and summarize a patient record from in-memory data
Input: patient_id: str
Output: patient_summary (string) or error

**match_guideline**

Purpose: Identify the most relevant clinical guideline for a given procedure and diagnosis using an LLM
Inputs: procedure: str, diagnosis: str
Output: matched_guideline (string)

**check_guideline_validity**

Purpose: Evaluate whether the patient’s symptoms and notes meet the guideline criteria
Inputs: symptoms: list[str], required_symptoms: list[str], notes: str
Output: validity_result (string)

**recommend_care_plan**

Purpose: Provide next-step recommendations based on the diagnosis
Input: diagnosis: str
Output: recommendation (string)

### Implementation Notes
All tools that use an LLM are powered by ChatOpenAI
Temperature is set to 0 for consistent, deterministic outputs
Streaming is enabled in the implementation
Each tool returns a single key with a concise textual explanation
Typical Execution Flow

The agent generally calls tools in this order:
fetch_patient_record(patient_id)
→ Retrieves and summarizes patient context
match_guideline(procedure, diagnosis)
→ Finds the most relevant clinical guideline
check_guideline_validity(symptoms, required_symptoms, notes)
→ Determines whether criteria are met or require further review
recommend_care_plan(diagnosis)
→ Suggests next steps or alternatives
Example Outputs

fetch_patient_record
```
{
  "patient_summary": "Patient ID: P102\nAge: 65, Sex: Female\nReported Symptoms: chest pain, shortness of breath, tachycardia\nPreliminary Diagnosis: Clinical suspicion of PE\nRequested Procedure: CT Chest\nClinical Notes: Wells score high probability; D-dimer positive."
}
```
match_guideline
```
{
  "matched_guideline": "CTPA is appropriate for high-probability PE with positive D-dimer. Required symptoms: chest pain, shortness of breath, tachycardia. Caveats: ensure renal function adequate for contrast."
}
```
check_guideline_validity
```
{
  "validity_result": "Criteria met: symptoms align and notes indicate high probability (Wells) with positive D-dimer. Imaging is medically necessary."
}
```
recommend_care_plan
```
{
  "recommendation": "Initiate anticoagulation, confirm with CTPA, monitor hemodynamics, and perform risk stratification."
}
```

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY)

@tool
def fetch_patient_record(patient_id: str) -> dict:
    """
    Fetches and summarizes a patient record based on the given patient ID.

    Returns a human-readable summary including age, sex, symptoms, diagnosis, procedure, and clinical notes.
    Also includes the raw patient record in case other tools or agents need structured access.

    Args:
        patient_id (str): The unique identifier for the patient.

    Returns:
        dict: {
            "summary": str,  # Natural language summary of the patient record
        }
    """
    for record in patient_records:
        if record["patient_id"] == patient_id:
            summary = (
                f"Patient ID: {record['patient_id']}\n"
                f"Age: {record['age']}, Sex: {record['sex']}\n"
                f"Reported Symptoms: {', '.join(record['symptoms'])}\n"
                f"Preliminary Diagnosis: {record['diagnosis']}\n"
                f"Requested Procedure: {record['procedure']}\n"
                f"Clinical Notes: {record['notes']}"
            )
            return {
                "patient_summary": summary
            }
    return {"error": "Patient record not found."}


@tool
def match_guideline(procedure: str, diagnosis: str) -> dict:
    """
    Match a given procedure and diagnosis to the most relevant clinical guideline.

    Args:
        procedure (str): The medical procedure being requested.
        diagnosis (str): The diagnosis related to the procedure.

    Returns:
        dict: A summary of the best matching guideline if found, or a message indicating no match.
    """
    context = "\n".join([
        f"{i+1}. Procedure: {g['procedure']}, Diagnosis: {g['diagnosis']}, Required Symptoms: {g['required_symptoms']}, Notes: {g['notes']}"
        for i, g in enumerate(medical_guidelines)])

    prompt = f"""You are a clinical reviewer assessing whether a requested medical procedure aligns with existing evidence-based guidelines.

Instructions:
- Analyze the patient's procedure and diagnosis.
- Compare against the list of provided clinical guidelines.
- Select the guideline that best fits the case by reasoning on the common matches considering procedure and diagnosis.
- If none match, respond: "No appropriate guideline found for this case."
- If a match is found, summarize the matching guideline clearly including any required symptoms or caveats.

Patient Case:
- Procedure: {procedure}
- Diagnosis: {diagnosis}

Available Guidelines:
{context}
"""
    result = llm.invoke(prompt).content
    return {"matched_guideline": result}


@tool
def check_guideline_validity(symptoms: list, required_symptoms: list, notes: str) -> dict:
    """
    Determine whether the patient's symptoms and notes satisfy the guideline criteria for medical necessity.

    Args:
        symptoms (list): List of symptoms recorded in the patient’s record.
        required_symptoms (list): List of symptoms required by the matched guideline.
        notes (str): Free-text clinical notes associated with the patient case.

    Returns:
        dict: A string with justification explaining whether the procedure is valid or not.
    """
    prompt = f"""You are validating a medical procedure request based on documented symptoms and clinical context.

Instructions:
- Assess whether the patient's symptoms and notes fulfill the required guideline criteria.
- Consider nuances or indirect references (e.g. "long flight" implies immobility).
- Provide a reasoned judgment if the procedure is medically necessary.
- If it does not qualify, explain exactly which criteria are unmet.

Input:
- Patient Symptoms: {symptoms}
- Required Symptoms from Guideline: {required_symptoms}
- Clinical Notes: {notes}
"""
    result = llm.invoke(prompt).content
    return {"validity_result": result}


@tool
def recommend_care_plan(diagnosis: str) -> dict:
    """
    Recommend a follow-up care plan based on a given diagnosis.

    Args:
        diagnosis (str): The diagnosis to evaluate for next steps.

    Returns:
        dict: A recommendation string describing the suggested care plan or a fallback message if no match is found.
    """
    options = "\n".join([
        f"{i+1}. Diagnosis: {c['diagnosis']}, Recommendation: {c['next_step']}"
        for i, c in enumerate(care_recommendations)])

    prompt = f"""You are a clinical support assistant suggesting appropriate next steps for a given medical diagnosis.

Instructions:
- Analyze the given diagnosis.
- Choose the closest match from the list of known recommendations.
- Explain why the match is appropriate.
- If no suitable recommendation is found, return: "No care recommendation found for this diagnosis."

Diagnosis Provided:
{diagnosis}

Available Recommendations:
{options}
"""
    result = llm.invoke(prompt).content
    return {"recommendation": result}

##Single-Agent System
![](https://i.imgur.com/s9hSJ6l.png)

1. Central Agent
Runs on a system prompt and an LLM. It manages patient intake, validates guidelines, and generates care recommendations within a single workflow. The agent determines when and how to use tools to analyze patient data and make decisions.

2. Tools Layer
The agent directly calls tools for specific functions:
  Fetch Patient Record: Retrieves patient history from the dataset
  Match Guideline: Identifies relevant medical guidelines for a diagnosis
  Check Guideline Validity: Confirms whether symptoms meet required criteria
  Fetch Recommendation: Provides suggested treatments or interventions

3. Datasets Layer
  Patient Records: Contains structured patient data
  Medical Guidelines: Stores diagnostic and procedural standards
  Care Recommendations: Includes treatment plans and care pathways

4. Final Output
The agent combines results from all tools to produce a final decision output, which includes decision status, reasoning, and recommended next steps.

In [ ]:
single_agent_prompt = """
You are a senior medical review assistant responsible for evaluating healthcare procedure requests.

You must call relevant tools to do the following:
1. Retrieve the full patient record using the patient ID.
2. Match the requested procedure and diagnosis to clinical guidelines.
3. Validate the match by comparing the patient's symptoms and notes to the guideline's requirements.
4. Recommend the appropriate next steps based on the diagnosis.
5. Output a final summary based on the guidelines given below.

Analyze all the results from the tool calls before making the final decision

Your final response should ONLY include the following bullets in the exact format specified:

- Final Decision: [APPROVED/NEEDS REVIEW]
- Decision Reasoning: [What criteria matched or did not match]
- Care recommendation or alternative steps: [care plan steps to take or alternative steps if it needs review]

Do NOT add any other extra content in the final response
"""
AGENT_SYS_PROMPT = SystemMessage(content= single_agent_prompt)

single_agent = create_react_agent(
    model = llm,
    tools = [fetch_patient_record, match_guideline, check_guideline_validity, recommend_care_plan],
    prompt = AGENT_SYS_PROMPT
)

In [ ]:
from IPython.display import Image
display(Image(single_agent.get_graph().draw_mermaid_png()))

### Stream Agent Execution

In [ ]:
from IPython.display import display, Markdown

def call_agent_system(agent, prompt, verbose=False):
    events = agent.stream(
        {"messages": [("user", prompt)]},
        {"recursion_limit": 25},
        stream_mode="values"
    )

    last_event = None

    for event in events:
        last_event = event
        if verbose:
            last_message = event["messages"][-1]
            print(last_message.content)

    if last_event is not None:
        print("\n\nFinal Response:")
        final_message = last_event["messages"][-1]
        display(Markdown(final_message.content))
    else:
        print("No response received from agent.")


prompt = "Review patient P101 for procedure justification."
call_agent_system(single_agent, prompt, verbose=True)

## Multi-Agent System with Supervisor (Healthcare Decision Support)

This system is designed to support healthcare decisions using multiple specialized agents coordinated by a central supervisor.

![](https://i.imgur.com/s9hSJ6l.png)

1. Supervisor Agent

The Supervisor Agent manages the overall workflow.
It uses a system prompt and a language model to decide which agent should handle each task.
It collects outputs from all agents and combines them into a final decision.

2. Specialized Agents

Clinical Intake Agent
Collects patient information.
Uses a tool to retrieve data from the Patient Records dataset.

Guideline Checker Agent
Checks whether care decisions follow medical guidelines.
Uses tools to match guidelines and verify their validity against the Medical Guidelines dataset.

Care Recommender Agent
Suggests treatments or interventions.
Uses a tool to retrieve recommendations from the Care Recommendations dataset.

Each agent operates using its own system prompt and language model.

3. Tools Layer

This layer allows agents to interact with external data sources.
It includes tools for:

Fetching patient records
Matching guidelines
Validating guidelines
Fetching care recommendations
4. Datasets Layer

The system relies on structured datasets:

Patient Records
Medical Guidelines
Care Recommendations

These datasets provide the information needed for agent reasoning.

5. Final Output

The Supervisor Agent gathers results from all agents and produces a final decision for clinical use.

### Implement Sub Agents

In [ ]:
clinical_intake_agent = create_react_agent(
    llm,
    tools=[fetch_patient_record],
    prompt=SystemMessage(content="""You are acting as a clinical intake specialist responsible for reviewing and summarizing a patient's medical case.

Your responsibilities:
1. Carefully read the symptoms, diagnosis, procedure, and notes with the right tool.
2. Generate a medically accurate clinical summary (key points).
3. Identify any additional risk factors or inferred clues from notes (e.g., “long flight” → immobility).
4. Derive the clinical rationale for why the procedure may have been ordered.

Your final message must clearly include:
- A clinical summary
- Key clinical findings (explicit or inferred)

Present your output as if it’s being passed to a medical reviewer for decision-making.
""")
)

guideline_checker_agent = create_react_agent(
    llm,
    tools=[match_guideline, check_guideline_validity],
    prompt=SystemMessage(content="""You are a utilization review specialist evaluating whether a requested procedure is medically justified.

Your responsibilities:
1. Identify the most relevant clinical guideline based on the procedure and diagnosis.
2. If a matching guideline is found, extract the required symptoms.
3. Then use an appropriate tool to check the guideline validity for the patient using:
   - the patient’s documented symptoms
   - the required symptoms from the guideline
   - clinical notes (from the intake summary)

Guidance:
- Use clinical reasoning to validate justification.
- If symptoms and context meet the criteria, justify the procedure clearly.
- If not, explain why the guideline requirements are not satisfied.
- If no matching guideline is found, state that clearly.

Your message should be clear, objective, and appropriate for escalation or approval review.
""")
)

care_recommender_agent = create_react_agent(
    llm,
    tools=[recommend_care_plan],
    prompt=SystemMessage(content="""
You are a clinical care assistant responsible for recommending follow-up actions based on a confirmed diagnosis.

Instructions:
1. Consider the outputs from the intake and guideline checker agents.
2. If the guideline checker has determined that the procedure is NOT justified, you must:
   - Suggest alternate steps (e.g., reassess symptoms, collect more data), OR
   - Justify why a care plan involving that procedure may still be warranted due to risk factors.
3. If the guideline checker approved the procedure, proceed with care planning as usual.

Important:
- Always clarify whether your recommendation is based on an approved procedure or despite a failed guideline check.
- Avoid assuming procedures are approved if guideline validation failed.

Use precise clinical reasoning.
""")
)


## Supervisor Agent Node Function

In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.types import Command

# Define shared state
class State(TypedDict):
    messages: Annotated[list, add_messages]

members = ["clinical_intake_agent", "guideline_checker_agent", "care_recommender_agent"]

SUPERVISOR_PROMPT = f"""You are a supervisor tasked with managing a healthcare utilization review conversation between the following agents:
{members}.

Each agent performs a specific part of the review process:
- clinical_intake_agent summarizes the patient's case and rationale
- guideline_checker_agent evaluates medical necessity against clinical guidelines
- care_recommender_agent suggests appropriate follow-up care

Read the current messages. Decide who should act next.
If the full workflow is complete, respond with FINISH.
"""

FINAL_RESPONSE_PROMPT = """Analyze all the results from the execution so far before making the final decision

Your final response should ONLY include the following bullets in the exact format specified:

- Final Decision: [APPROVED/NEEDS REVIEW]
- Decision Reasoning: [What criteria matched or did not match]
- Care recommendation or alternative steps: [care plan steps to take or alternative steps if it needs review]

Do NOT add any other extra content in the final response
"""

class Router(TypedDict):
    next: Literal["clinical_intake_agent", "guideline_checker_agent", "care_recommender_agent", "FINISH"]

def supervisor_node(state: State) -> Command[Literal["clinical_intake_agent",
                                                     "guideline_checker_agent",
                                                     "care_recommender_agent",
                                                     "__end__"]]:
    # Figure out which sub-agent should act next
    messages = [SystemMessage(content=SUPERVISOR_PROMPT)] + state["messages"]
    response = llm.with_structured_output(Router).invoke(messages)
    goto = response["next"]
    # If all agents finished - process context and generate final response
    if goto == "FINISH":
        goto = END
        messages = [SystemMessage(content=FINAL_RESPONSE_PROMPT)] + state["messages"]
        response = llm.invoke(messages)

        return Command(goto=goto, update={"messages": [AIMessage(content=response.content,
                                                                name="supervisor")],
                                          "next": goto})

    return Command(goto=goto, update={"next": goto})

## Sub-Agents Node Functions


In [ ]:
def clinical_intake_node(state: State) -> Command[Literal["supervisor"]]:
    result = clinical_intake_agent.invoke(state)
    return Command(
        update={"messages": [AIMessage(content=result["messages"][-1].content,
                                          name="clinical_intake_agent")]},
        goto="supervisor"
    )

def guideline_checker_node(state: State) -> Command[Literal["supervisor"]]:
    result = guideline_checker_agent.invoke(state)
    return Command(
        update={"messages": [AIMessage(content=result["messages"][-1].content,
                                          name="guideline_checker_agent")]},
        goto="supervisor"
    )

def care_recommender_node(state: State) -> Command[Literal["supervisor"]]:
    result = care_recommender_agent.invoke(state)
    return Command(
        update={"messages": [AIMessage(content=result["messages"][-1].content,
                                          name="care_recommender_agent")]},
        goto="supervisor"
    )

 ## Multi-Agent Graph

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_edge(START, "supervisor")
graph_builder.add_node("supervisor", supervisor_node)
graph_builder.add_node("clinical_intake_agent", clinical_intake_node)
graph_builder.add_node("guideline_checker_agent", guideline_checker_node)
graph_builder.add_node("care_recommender_agent", care_recommender_node)

multi_agent = graph_builder.compile()
display(Image(multi_agent.get_graph(xray=True).draw_mermaid_png()))

### Stream Agent Exeuction

In [ ]:
prompt = "Review patient P101 for procedure justification."
call_agent_system(multi_agent, prompt, verbose=False)

## Test Execution Times of Single vs. Multi-Agent

In [ ]:
def run_review():
    prompt = "Review patient P102 for procedure justification."
    call_agent_system(single_agent, prompt, verbose=False)

%timeit -n 1 -r 3 run_review()

In [ ]:
def run_review():
    prompt = "Review patient P102 for procedure justification."
    call_agent_system(multi_agent, prompt, verbose=False)

%timeit -n 1 -r 3 run_review()